# 第2回：Pythonを読み、Copilotと少し変える

**今日の問い：分からないコードを、どうやって小さく理解し、安全に書き換えるか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 変数・リスト・辞書・条件分岐・繰り返し・関数を読める
- 型ヒント・docstring・防御的な入力検査を備えた関数を書く
- assertによる小さなテストで、境界値と例外を先に固定する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 型ヒント：引数と戻り値の型を明示する注釈
- docstring：関数の目的と使い方を書く文字列
- 例外：処理を続けられない理由を伝える仕組み
- 単体テスト：関数の入出力を自動で確かめる小さなコード
- 純粋関数：同じ入力へ常に同じ出力を返し副作用のない関数

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## なぜ「読む」練習から始めるのか

これからの回では、完成したコードを**少しだけ書き換えて**実験します。ゼロから書けなくても、
**読めて・1か所いじれる**ようになれば十分に前へ進めます。この回は、以後ずっと出てくる
4つの部品（値・リスト・辞書・繰り返し・関数）に絞って読み方を身につけます。文法の網羅はしません。

Copilotは「一発で完成品を作らせる道具」ではなく、「短い相談を何度もする相棒」として使います。
提案は必ず1つずつ試し、出力を自分の目で確かめます。


## 変数・リスト・辞書：データを入れる3つの箱

- **変数**：1つの値に名前を付けた箱（例：`sample_name`）。
- **リスト `[...]`**：順番のある複数の値（例：温度の並び）。
- **辞書 `{key: value}`**：名前で値を引く箱（例：`experiment["solvent"]`で溶媒を取り出す）。

`type(x)`は「その値が何型か」を教えてくれます。実行して、3つの箱の見た目の違いを確かめましょう。


In [ ]:
sample_name = "CMP-0001"
temperatures = [60, 75, 90]
experiment = {"sample_id": sample_name, "solvent": "EtOH", "active": 1}
print(type(sample_name), sample_name)
print(type(temperatures), temperatures)
print(type(experiment), experiment)


### 出力の読み方

- `<class 'str'>`は**文字列**、`<class 'list'>`は**リスト**、`<class 'dict'>`は**辞書**。
- 辞書は`{'sample_id': 'CMP-0001', ...}`のように、**名前（キー）と値**の組で並びます。
- pandasの表（`df`）は、ざっくり言うと「辞書（列名→列の値）」と「リスト（行の並び）」を合わせたものです。この3つが分かると表データも読みやすくなります。


## TRY：`for`（繰り返し）と`if`（条件分岐）を読む

`for`は「リストの要素を1つずつ取り出して同じ処理を繰り返す」書き方、`if ... else`は
「条件で処理を分ける」書き方です。**実行する前に、何行表示されるか予想**してから動かしましょう。
予想と結果を比べるのが、コードを読む力を最短で伸ばすコツです。


In [ ]:
for temperature in temperatures:
    label = "高温条件" if temperature >= 75 else "低温条件"
    print(temperature, label)


### 出力の読み方

- `temperatures`は3要素なので**3行**出ます（予想は合っていましたか？）。
- 各行で`temperature`が60→75→90と変わり、`75以上か`で「高温／低温」が切り替わります。
- `A if 条件 else B`は「条件が真ならA、偽ならB」を1行で書く形。`if:` を複数行で書いても同じ意味です。


## 関数：処理に名前を付けて再利用する

**関数**は「入力を受け取り、決まった処理をして、結果を返す」部品です。同じ計算を何度も書かずに済みます。

- `def 関数名(引数: 型) -> 戻り値の型:` の**型ヒント**は、読み手（と生成AI）への注釈です。動作は変えませんが、誤解を減らします。
- 直後の文字列は**docstring**（関数の説明）。`help(関数)`で読めます。
- `[celsius_to_kelvin(v) for v in temperatures]`は**リスト内包表記**。「各要素に関数をかけた新しいリスト」を1行で作ります。


In [ ]:
def celsius_to_kelvin(celsius: float) -> float:
    "摂氏をケルビンへ変換する。"
    return celsius + 273.15

converted = [celsius_to_kelvin(value) for value in temperatures]
print(converted)
help(celsius_to_kelvin)


### 出力の読み方

- `converted`は、各温度に273.15を足したリスト（例：`[333.15, 348.15, 363.15]`）。
- `help(...)`は、書いておいたdocstringと引数の形を表示します。**自分の関数にも説明が付く**ことを体験しておきましょう。


## TRY：エラーは「読む」もの。省略せず全文を見る

エラーは失敗ではなく、**どこで何が起きたかの手がかり**です。わざと存在しない要素を取り出して、
エラーの形を観察します。`try/except`は「エラーが出ても止まらず、内容を受け取る」書き方です。


In [ ]:
try:
    temperatures[10]
except Exception as error:
    print(type(error).__name__)
    print(error)


### 出力の読み方

- `IndexError`という**エラーの種類（名前）**と、`list index out of range`という**説明**が出ます。
- リストは0番から数えるので、3要素の`temperatures`に`[10]`は存在せず、範囲外エラーになります。
- 実際のエラーでは、**末尾の1〜2行**（種類とメッセージ）にいちばん近い原因が書かれています。Copilotに貼るときも、この全文を省略しないことが大切です。


## CHANGE

`temperatures`へ温度を1つ追加し、`for`ループと変換結果の表示がどう変わるか確認します。

## ASK COPILOT

気になるセルを貼り、「各行の実行後に、変数の型と中身がどう変わるか表で説明して」と依頼します。
提案は1つずつ試し、必ず出力で答え合わせをします。


## DEEP DIVE：テストで守る小さなユーティリティ

ここからは発展です。「実行できる」ことと「正しい」ことは別物です。特にCopilotが書いたコードは、
**普通の入力では動いても、変な入力で静かに間違える**ことがあります。そこで、**変な入力を先に想定して
弾く関数**を書きます。

- `raise TypeError(...)` / `raise ValueError(...)` は、「この入力は受け付けない」と**わざとエラーを起こす**書き方。
- こうしておくと、間違った使い方をした人にすぐ気づいてもらえます（沈黙して誤った答えを返すより安全）。


In [ ]:
def celsius_to_kelvin_checked(celsius: float) -> float:
    "型と物理的な下限を検査してから摂氏をケルビンへ変換する。"
    if not isinstance(celsius, (int, float)):
        raise TypeError("温度は数値で入力してください")
    if celsius < -273.15:
        raise ValueError("絶対零度より低い値は指定できません")
    return celsius + 273.15

for value in [25, -273.15, -300, "25"]:
    try:
        print(value, "->", round(celsius_to_kelvin_checked(value), 2))
    except (TypeError, ValueError) as error:
        print(value, "->", type(error).__name__, error)


### 出力の読み方

4つの入力それぞれの結果が並びます。`25`は正常変換、`-273.15`は境界（絶対零度ちょうど）でOK、
`-300`は物理的にありえないので`ValueError`、`"25"`は数値でなく文字列なので`TypeError`。
**正常・境界・異常**を1度に確かめられました。


### assertで「期待する答え」を先に書いて固定する

`assert 式` は「式が真でなければ止まれ」という自己点検でした。これを使うと、関数の**テスト**が書けます。
コツは、**答えを先に書いてから**関数を作ること。ここではIQR法（四分位範囲）で外れ値を除く関数を、
正常・空リスト・NaN混在の3ケースで検証します。


In [ ]:
import numpy as np

def drop_outliers_iqr(values: list[float], k: float = 1.5) -> list[float]:
    "IQR法で外れ値を除いた値のリストを返す。NaNは事前に除く。"
    clean = [v for v in values if v == v]  # NaN(v != v)を除外
    if not clean:
        return []
    q1, q3 = np.percentile(clean, [25, 75])
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return [v for v in clean if low <= v <= high]

assert drop_outliers_iqr([10, 11, 12, 13, 1000]) == [10, 11, 12, 13]
assert drop_outliers_iqr([]) == []
assert drop_outliers_iqr([5, 5, float("nan")]) == [5, 5]
print("すべてのテストを通過しました")


### 出力の読み方

- 3つの`assert`がすべて通ると、最後の`print`だけが表示されます。**エラーが出ない＝合格**です。
- もし関数を壊すと（例：`k`を0にする）、どの`assert`で止まったかが表示され、**間違いの場所がすぐ分かります**。
- `v == v`が`False`になるのはNaNだけ、という小技で欠損を除いています。


## CHALLENGE：関数の「性質」を調べる

外れ値を除いた後、もう一度同じ関数をかけると、さらに減るでしょうか。1回目で四分位が変わるため、
**必ずしも同じ結果（冪等）にはなりません**。こうした「関数の性質」を意識すると、思わぬ副作用に気づけます。


In [ ]:
rng = np.random.default_rng(0)
sample = rng.normal(50, 5, 200).tolist()
once = drop_outliers_iqr(sample)
twice = drop_outliers_iqr(once)
print("1回適用後の件数:", len(once))
print("2回目でさらに減った件数:", len(once) - len(twice))
print("2回目で変化なし(冪等):", once == twice)


### 出力の読み方

`2回目でさらに減った件数`が0でなければ、この関数は**冪等ではない**（適用回数で結果が変わる）と分かります。
外れ値除去を繰り返し適用する前処理は、この性質のせいで「消しすぎ」が起きやすい、という教訓につながります。


## APPENDIX（任意・追加演習）

90分の外で、以後よく使うPythonの部品をもう少し練習します。飛ばしても本編は進められます。
まずは`enumerate`（番号付き繰り返し）・`zip`（同時に回す）・`sorted`（並べ替え）・条件付き内包表記です。


In [ ]:
samples = ["CMP-0001", "CMP-0002", "CMP-0003"]
yields = [82.5, 40.1, 63.7]

for i, name in enumerate(samples, start=1):        # 番号付きで回す
    print(i, name)

pairs = {name: y for name, y in zip(samples, yields)}   # 2つを同時に回して辞書化
print("辞書:", pairs)

ranked = sorted(pairs.items(), key=lambda kv: kv[1], reverse=True)  # 収率降順
print("収率降順:", ranked)

high = [name for name, y in pairs.items() if y >= 60]   # 条件付き内包表記
print("収率60以上:", high)


### 出力の読み方

- `enumerate`は`(番号, 要素)`を返すので、行番号付きの表示に便利。
- `zip`は複数リストを同時に回します。`for a, b in zip(...)`の形は頻出です。
- `sorted(..., key=..., reverse=True)`で並べ替え。`key`に「何で並べるか」を関数で渡します。
- これらは`for`ループを短く読みやすくする道具で、pandasの内部でも同じ発想が使われています。


### 自分の「状態を持つ部品」を作る（クラス入門）

関数は入力→出力の1回きりですが、**クラス**は状態を持ち続けられます。値を足しながら件数と平均を
保つ小さなクラスを書き、`assert`で動作を確かめます。難しければ「こういう書き方がある」で十分です。


In [ ]:
class RunningStats:
    "値を1つずつ足しながら件数・合計・平均を保つ小さなクラス。"
    def __init__(self):
        self.n = 0
        self.total = 0.0
    def add(self, value: float) -> None:
        self.n += 1
        self.total += value
    @property
    def mean(self) -> float:
        return self.total / self.n if self.n else float("nan")

stats = RunningStats()
for y in [82.5, 40.1, 63.7]:
    stats.add(y)
assert stats.n == 3
assert abs(stats.mean - 62.1) < 0.1
print(f"件数={stats.n} 平均={stats.mean:.1f}")


### 出力の読み方

`add`を呼ぶたびに内部の`n`と`total`が更新され、`mean`はいつでも現在の平均を返します。`assert`が通れば
実装は期待どおり。scikit-learnのモデルも「`fit`で状態を覚え、`predict`で使う」クラスなので、この
仕組みが分かると内部のイメージがつかめます。


### pandasに橋渡しする

第3回で本格的に使うpandasを、ひと足先に少しだけ触ります。CSVを読み、1列（Series）の平均や
種類を取り出します。


In [ ]:
import pandas as pd

data = pd.read_csv(DATA / "compound_experiments.csv")
print("1列の型:", type(data["yield_pct"]).__name__)
print("平均収率:", round(data["yield_pct"].mean(), 1))
print("溶媒の種類:", data["solvent"].dropna().unique().tolist())


### 出力の読み方

`data["yield_pct"]`は1列（Series）で、`.mean()`のような集計をそのまま呼べます。`.unique()`は値の種類、
`.dropna()`は欠損を除く指定。ここまで来れば、第3回のpandasはぐっと読みやすくなります。


## よくある誤り

- Notebookを途中から実行して変数がない
- Copilotの長い修正を一度に採用する
- エラー全文を読まずにセルを繰り返し実行する

## SELF-STUDY（任意・30〜60分）

- 収率のリストから外れ値をIQRで除く関数を、型ヒントとテスト付きで書く
- 正常値・空リスト・NaN混在の3ケースを、期待結果を先に書いてから検証する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 型ヒントとdocstringは何の役に立つか
2. assertは何を保証し、何を保証しないか
3. 生成AIのコードを何で確認するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
